# HairFastGAN — 4 trials เริ่มจากโมเดลต้นฉบับที่ยังไม่จูนด้วยข้อมูลของเรา

**ฉบับนี้ใช้แทน OriginalStart และ FromEpoch27**.
ทุก trial โหลด **Blending checkpoint ต้นฉบับของ HairFastGAN + PostProcess checkpoint ต้นฉบับ** ใหม่ ไม่โหลด uniform epoch02 และไม่ฝึกต่อจาก Full150 epoch27.
นี่คือ pretrained weights ของผู้พัฒนา ไม่ใช่การสุ่มน้ำหนักใหม่/ฝึก GAN จากศูนย์.

| Trial | Blending LR | PostProcess LR |
|---|---:|---:|
| A | 1e-5 | 1e-6 |
| B | 1e-5 | 5e-7 |
| C | 5e-6 | 1e-6 |
| D | 5e-6 | 5e-7 |

- จุดเริ่ม A–D: official pretrained Blending + official pretrained PostProcess เหมือนกันทุกชุด; optimizer ใหม่
- คง Alignment และ cache/masks/150 train/9 validation เดิม, loss เดิม, batch=1, seed=3407
- **30 epochs ต่อ trial / 4,500 updates** รวม 18,000 updates
- **epoch0** = โมเดลต้นฉบับก่อนจูนด้วยข้อมูลของคุณ
- **old ep27 / epoch -27 ใน CSV** = ผล Full150 เดิมที่ประเมินเปรียบเทียบเท่านั้น; คืนค่าน้ำหนักต้นฉบับก่อนเริ่มฝึก
- A ใช้ LR เดิม แต่ไม่ใช่การทำซ้ำ Full150 เดิมทุกเงื่อนไข เพราะ Full150 เดิมเริ่มจาก Blending ที่เคยจูน 2 epochs

เลือก best ภายในแต่ละ trial เทียบ epoch0 ของโมเดลต้นฉบับ แล้วนำ best A–D เทียบ old epoch27 ด้วย masks/metrics เดียวกัน พร้อมภาพ. หากไม่มีตัวใหม่ผ่านเกณฑ์ชนะ epoch27 ยังเลือกผลเดิมได้.

เปิด **Colab runtime ใหม่**, เลือก GPU แล้ว Run all. ใช้ Drive cache/masks เดิมและ Full150 best.pt สำหรับเปรียบเทียบ. ไม่ต้องมีไฟล์ uniform epoch02.
ผลอยู่ใต้ **HairFastGAN_BPP_4Trials_PretrainedStart**. ห้าม resume จาก trials รุ่นก่อน; RESUME_STUDY_DIR ต้องเป็นรอบ PretrainedStart นี้เท่านั้น.

ตรวจ syntax และทดสอบตรรกะเริ่มจาก pretrained/คืนค่าหลังประเมินแล้ว แต่ยังไม่ได้รันโมเดลจริง/GPU.


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('กรุณาเลือก Runtime > Change runtime type > T4 GPU แล้ว Run all อีกครั้ง')
print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
%pip install -q opencv-python matplotlib scipy scikit-image scikit-learn pandas tqdm dlib face-alignment gdown addict wandb torchmetrics dill ninja transformers diffusers accelerate fastapi uvicorn python-multipart pyngrok nest-asyncio huggingface_hub
%pip install -q git+https://github.com/openai/CLIP.git
import torch
assert torch.cuda.is_available(), 'เลือก Runtime > Change runtime type > GPU ก่อนรัน'
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)


In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil, subprocess, sys, json, time
from huggingface_hub import snapshot_download

drive.mount('/content/drive')
LOCAL_REPO = '/content/HairFastGAN'
DRIVE_CACHE_DIR = Path('/content/drive/MyDrive/HairFastGAN_cache/pretrained_models')
RESULTS_ROOT = Path('/content/drive/MyDrive/HairFastGAN_prompt_study')
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
# ใส่ commit ที่บันทึกใน metadata.json เมื่อต้องการทำซ้ำภายหลัง
HAIRFAST_REVISION = '49e98019c9243d1991c19d45764fc3340a498de2'
if not Path(LOCAL_REPO, '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AIRI-Institute/HairFastGAN.git', LOCAL_REPO], check=True)
if HAIRFAST_REVISION:
    subprocess.run(['git', '-C', LOCAL_REPO, 'checkout', HAIRFAST_REVISION], check=True)
os.chdir(LOCAL_REPO)
local_models = Path(LOCAL_REPO, 'pretrained_models')
if not local_models.exists() or not any(local_models.iterdir()):
    if DRIVE_CACHE_DIR.exists() and any(DRIVE_CACHE_DIR.iterdir()):
        shutil.copytree(DRIVE_CACHE_DIR, local_models, dirs_exist_ok=True)
    else:
        snapshot = snapshot_download('AIRI-Institute/HairFastGAN', allow_patterns=['pretrained_models/**'])
        shutil.copytree(Path(snapshot, 'pretrained_models'), local_models, dirs_exist_ok=True)
        shutil.copytree(local_models, DRIVE_CACHE_DIR, dirs_exist_ok=True)
# ตรวจ cache สำคัญ รวมถึงกรณีได้ Git-LFS pointer แทน weights
for relative in ['StyleGAN/ffhq.pt', 'encoder4editing/e4e_ffhq_encode.pt']:
    p = local_models / relative
    if not p.exists() or p.stat().st_size < 1000000:
        raise RuntimeError(f'โมเดลขาดหรือไม่สมบูรณ์: {p} โปรดตรวจ cache ก่อนดำเนินการ')
Path('input').mkdir(exist_ok=True)
Path('output').mkdir(exist_ok=True)
REPO_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Repo commit:', REPO_COMMIT)


In [ ]:
# Compatibility backend: native torch operators also execute on CUDA tensors.
# No PyTorch/CUDA version downgrade; no changes to weights or transfer parameters.
import ast, hashlib, importlib, sys
from pathlib import Path
STYLEGAN_OPS_BACKEND = 'torch_native'

def _native_stylegan_source(source, kind):
    marker = '# HAIRFAST_TORCH_NATIVE_V1'
    if marker in source:
        return source
    tree = ast.parse(source)
    extension = 'fused' if kind == 'fused_act' else 'upfirdn2d_op'
    entry = 'fused_leaky_relu' if kind == 'fused_act' else 'upfirdn2d'
    edits = []
    found_load = found_entry = False
    for node in tree.body:
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == extension for t in node.targets):
            if not isinstance(node.value, ast.Call) or not isinstance(node.value.func, ast.Name) or node.value.func.id != 'load':
                raise RuntimeError('รูปแบบ extension เปลี่ยนไป: ' + kind)
            edits.append((node.lineno-1, node.end_lineno, extension + ' = None\n'))
            found_load = True
        if isinstance(node, ast.FunctionDef) and node.name == entry:
            edits.append((node.lineno-1, node.end_lineno, FUSED_NATIVE if kind == 'fused_act' else UP_NATIVE))
            found_entry = True
    if not found_load or not found_entry:
        raise RuntimeError('ไม่พบจุด patch ที่คาดไว้ใน ' + kind)
    lines = source.splitlines(True)
    for start,end,text in sorted(edits, reverse=True):
        lines[start:end] = [text]
    updated = marker + '\n' + ''.join(lines)
    if kind == 'upfirdn2d' and not any(isinstance(n, ast.FunctionDef) and n.name == 'upfirdn2d_native' for n in tree.body):
        updated += '\n\n' + UP_NATIVE_IMPL + '\n'
    compile(updated, kind + '.py', 'exec')
    return updated

FUSED_NATIVE = 'def fused_leaky_relu(input, bias=None, negative_slope=0.2, scale=2 ** 0.5):\n    if bias is not None:\n        input = input + bias.reshape(1, -1, *([1] * (input.ndim - 2)))\n    return torch.nn.functional.leaky_relu(input, negative_slope=negative_slope) * scale\n'
UP_NATIVE = 'def upfirdn2d(input, kernel, up=1, down=1, pad=(0, 0)):\n    return upfirdn2d_native(input, kernel, up, up, down, down,\n                           pad[0], pad[1], pad[0], pad[1])\n'
UP_NATIVE_IMPL = 'def upfirdn2d_native(\n    input, kernel, up_x, up_y, down_x, down_y, pad_x0, pad_x1, pad_y0, pad_y1\n):\n    _, channel, in_h, in_w = input.shape\n    input = input.reshape(-1, in_h, in_w, 1)\n\n    _, in_h, in_w, minor = input.shape\n    kernel_h, kernel_w = kernel.shape\n\n    out = input.view(-1, in_h, 1, in_w, 1, minor)\n    out = F.pad(out, [0, 0, 0, up_x - 1, 0, 0, 0, up_y - 1])\n    out = out.view(-1, in_h * up_y, in_w * up_x, minor)\n\n    out = F.pad(\n        out, [0, 0, max(pad_x0, 0), max(pad_x1, 0), max(pad_y0, 0), max(pad_y1, 0)]\n    )\n    out = out[\n        :,\n        max(-pad_y0, 0) : out.shape[1] - max(-pad_y1, 0),\n        max(-pad_x0, 0) : out.shape[2] - max(-pad_x1, 0),\n        :,\n    ]\n\n    out = out.permute(0, 3, 1, 2)\n    out = out.reshape(\n        [-1, 1, in_h * up_y + pad_y0 + pad_y1, in_w * up_x + pad_x0 + pad_x1]\n    )\n    w = torch.flip(kernel, [0, 1]).view(1, 1, kernel_h, kernel_w)\n    out = F.conv2d(out, w)\n    out = out.reshape(\n        -1,\n        minor,\n        in_h * up_y + pad_y0 + pad_y1 - kernel_h + 1,\n        in_w * up_x + pad_x0 + pad_x1 - kernel_w + 1,\n    )\n    out = out.permute(0, 2, 3, 1)\n    out = out[:, ::down_y, ::down_x, :]\n\n    out_h = (in_h * up_y + pad_y0 + pad_y1 - kernel_h) // down_y + 1\n    out_w = (in_w * up_x + pad_x0 + pad_x1 - kernel_w) // down_x + 1\n\n    return out.view(-1, channel, out_h, out_w)'

def install_native_stylegan_ops(repo):
    root = Path(repo).resolve()
    pairs = sorted((root / 'models').rglob('op/fused_act.py'))
    if not pairs:
        raise RuntimeError('ไม่พบ StyleGAN operator ใน ' + str(root))
    planned = []
    for fused_path in pairs:
        for path,kind in [(fused_path,'fused_act'), (fused_path.with_name('upfirdn2d.py'),'upfirdn2d')]:
            if not path.is_file():
                raise RuntimeError('ขาด operator: ' + str(path))
            module_name = '.'.join(path.relative_to(root).with_suffix('').parts)
            if module_name in sys.modules:
                raise RuntimeError('StyleGAN ถูก import แล้ว: Restart runtime แล้วรันตามลำดับใหม่')
            old = path.read_text()
            planned.append((path, old, _native_stylegan_source(old,kind)))
    # Validate every candidate before writing; preserve each original once.
    records = []
    for path,old,new in planned:
        if old != new:
            backup = path.with_suffix('.py.before_native')
            if not backup.exists(): backup.write_text(old)
            path.write_text(new)
        records.append({'path':str(path.relative_to(root)),
                        'sha256':hashlib.sha256(new.encode()).hexdigest()})
    importlib.invalidate_caches()
    return records

STYLEGAN_NATIVE_PATCHES = install_native_stylegan_ops(LOCAL_REPO)
print('StyleGAN backend:', STYLEGAN_OPS_BACKEND, '| patched files:', len(STYLEGAN_NATIVE_PATCHES))

# Small on-device smoke check before loading model weights.
import torch
_check = {'torch':torch, 'F':torch.nn.functional}
exec(FUSED_NATIVE + '\n' + UP_NATIVE_IMPL + '\n' + UP_NATIVE, _check)
_device = 'cuda' if torch.cuda.is_available() else 'cpu'
_x = torch.randn(1, 2, 5, 5, device=_device, requires_grad=True)
_b = torch.randn(2, device=_device, requires_grad=True)
_y = _check['fused_leaky_relu'](_x, _b, .1)
torch.testing.assert_close(_y, torch.nn.functional.leaky_relu(_x + _b.view(1,2,1,1), .1) * (2 ** .5))
_z = _check['upfirdn2d'](_x, torch.ones(1,1,device=_device))
torch.testing.assert_close(_z, _x)
(_y.sum() + _z.sum()).backward()
assert torch.isfinite(_x.grad).all() and torch.isfinite(_b.grad).all()
print('Native operator smoke check passed on', _device)
del _check, _x, _b, _y, _z


In [ ]:
from PIL import Image
if not hasattr(Image,"ANTIALIAS"): Image.ANTIALIAS=Image.Resampling.LANCZOS
import sys, torch
sys.path.insert(0, LOCAL_REPO)
from hair_swap import HairFast, get_parser
hair_fast = HairFast(get_parser().parse_args([]))
print('HairFastGAN พร้อม')


## ข้อมูลเดิมและค่าทดลอง

In [ ]:
# A/B/C/D restart from OFFICIAL PRETRAINED weights; epoch27 is comparator only.
EPOCHS=30
SEED=3407
RESUME_STUDY_DIR='' # เมื่อ Colab หลุด: ใส่ STUDY_ROOT เดิม แล้ว Run all ใน runtime ใหม่
COMPARATOR_PT='/content/drive/MyDrive/HairFastGAN_Full150_30Epoch/0637a32ae8/best.pt'
TRIALS=[dict(name='A',blend_lr=1e-5,pp_lr=1e-6),
        dict(name='B',blend_lr=1e-5,pp_lr=5e-7),
        dict(name='C',blend_lr=5e-6,pp_lr=1e-6),
        dict(name='D',blend_lr=5e-6,pp_lr=5e-7)]
import io,csv,zipfile,hashlib,uuid,random,copy,traceback,time,gc
import numpy as np
import pandas as pd
import torch.nn.functional as F
from PIL import Image,ImageDraw
from pathlib import PurePosixPath
from google.colab import files
from IPython.display import display
from models.Net import get_segmentation as segment
from torch.utils.checkpoint import checkpoint

def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
SOURCE=Path('/content/drive/MyDrive/HairFastGAN_Expanded_Train_Preflight/5eb3a720c719_v1')
SOURCE_MANIFEST_SHA='f615c81e7ca885a03d83888ff4f772fd2993e129b17a48168e36ecb482799f18'
START_SHA='cece9608f2fe8269796a9903fb9550f2cb0e8022cedaefcbf9972305f206fed1' # official pretrained Blending
PP_START_SHA='91ae95d3bf274a61d4d120daa955346033ce089c334221d5b444ff2ce92a6a84'
MASK_ZIP_SHA='dd75d40ccf2db8239b5629f96b4d802623d68b716efdc196df3fd1194791aa37'
AUDIT_ZIP_SHA='fd9c1593d9ae2219727ff7c8120e5fc538a3f31d8d3d34ed5e1c6f5bd8a7be8b'
START=Path(hair_fast.blend.opts.blending_checkpoint)
assert sha(START)==START_SHA,'Official pretrained Blending mismatch'
assert sha(hair_fast.blend.opts.pp_checkpoint)==PP_START_SHA,'Official pretrained PostProcess mismatch'
REFERENCE_PT=Path(COMPARATOR_PT);REFERENCE_SHA=sha(REFERENCE_PT)
assert REFERENCE_SHA=='34e72cc83f955680fd59276bf1104620bea82534b4e3a524ed5aa9199fc1480d','epoch27 hash mismatch'
REFERENCE=torch.load(REFERENCE_PT,map_location='cpu',weights_only=True)
assert REFERENCE['epoch']==27 and REFERENCE['format']=='hairfast_joint_full150_v1'
assert REFERENCE['repo_commit']==REPO_COMMIT=='49e98019c9243d1991c19d45764fc3340a498de2'
assert REFERENCE['source_manifest_sha256']==SOURCE_MANIFEST_SHA and REFERENCE['mask_zip_sha256']==MASK_ZIP_SHA
assert sha(SOURCE/'manifest.json')==SOURCE_MANIFEST_SHA
meta=json.loads((SOURCE/'manifest.json').read_text());cases=json.loads((SOURCE/'cases.json').read_text())
assert cases==meta['cases']
bykey={c['key']:c for c in cases}
TRAIN_KEYS=[c['key'] for c in cases if c['split']=='train']
VAL_KEYS=[c['key'] for c in cases if c['split']=='validation'];KEYS=TRAIN_KEYS+VAL_KEYS
assert len(TRAIN_KEYS)==150 and len(VAL_KEYS)==9 and len(set(KEYS))==159
assert TRAIN_KEYS==REFERENCE['config']['train_keys'] and VAL_KEYS==REFERENCE['config']['validation_keys']
for role in ['face','shape','color']:
    assert {bykey[k][role] for k in TRAIN_KEYS}.isdisjoint({bykey[k][role] for k in VAL_KEYS})
STUDY_ROOT=Path(RESUME_STUDY_DIR) if RESUME_STUDY_DIR else Path('/content/drive/MyDrive/HairFastGAN_BPP_4Trials_PretrainedStart')/uuid.uuid4().hex[:10]
if RESUME_STUDY_DIR and not (STUDY_ROOT/'study.json').is_file():raise FileNotFoundError('ไม่พบ study.json ของรอบที่จะ resume')
STUDY_ROOT.mkdir(parents=True,exist_ok=True)
study_config=dict(version='bpp_4trials_pretrained_start_v1',start_blending_sha256=START_SHA,start_postprocess_sha256=PP_START_SHA,init_postprocess='original_pretrained',trials=TRIALS,epochs=EPOCHS,seed=SEED,batch_size=1,
    comparator_sha256=REFERENCE_SHA,source_manifest_sha256=SOURCE_MANIFEST_SHA,mask_zip_sha256=MASK_ZIP_SHA,
    repo_commit=REPO_COMMIT)
if (STUDY_ROOT/'study.json').exists():
    assert json.loads((STUDY_ROOT/'study.json').read_text())==study_config,'Study config changed'
else:(STUDY_ROOT/'study.json').write_text(json.dumps(study_config,indent=2))
print('STUDY_ROOT (เก็บ path นี้สำหรับ resume):',STUDY_ROOT)

# Reuse reviewed masks from the exact Full150 run. No new mask/crop generation.
MASKS=REFERENCE_PT.parent/'reviewed_masks'
if not (MASKS/'expanded_review_plan.json').is_file():
    MASKS=STUDY_ROOT/'reviewed_masks'
    if not (MASKS/'expanded_review_plan.json').is_file():
        print('ไม่พบ masks ใน Drive: อัปโหลด expanded_reviewed_color_masks (4).zip เดิม')
        up=files.upload()
        if len(up)!=1:raise ValueError('เลือก ZIP mask เดิมหนึ่งไฟล์')
        blob=next(iter(up.values()))
        if hashlib.sha256(blob).hexdigest()!=MASK_ZIP_SHA:raise ValueError('Mask ZIP hash mismatch')
        with zipfile.ZipFile(io.BytesIO(blob)) as z:
            if sum(i.file_size for i in z.infolist())>20*1024**2:raise ValueError('Unexpected ZIP size')
            for i in z.infolist():
                p=PurePosixPath(i.filename)
                if p.is_absolute() or '..' in p.parts or ((i.external_attr>>16)&0o170000)==0o120000:raise ValueError('Unsafe ZIP')
            MASKS.mkdir(exist_ok=True);z.extractall(MASKS)
        del blob,up
plan=json.loads((MASKS/'expanded_review_plan.json').read_text())
REVIEWED_KEYS=set(plan['train_keys']+plan['validation_keys'])
assert len(plan['train_keys'])==24 and len(REVIEWED_KEYS)==33 and set(plan['validation_keys'])==set(VAL_KEYS)
for k in REVIEWED_KEYS:
    rec=json.loads((MASKS/k/'decision.json').read_text())
    assert rec['case']==k and rec['status']=='human_reviewed' and rec['split']==bykey[k]['split'] and rec['audit_sha256']==plan['audit_sha256']
    for name,field in [('color_core.png','color_core_sha256'),('reference_core.png','reference_core_sha256'),('target_unchanged.png','original_target_sha256')]:
        assert sha(MASKS/k/name)==rec[field],(k,name)
# Bind current mask bytes to the study, protecting resumes against edits to masks/decisions.
mask_files={str(p.relative_to(MASKS)):sha(p) for p in MASKS.rglob('*') if p.is_file()}
if (STUDY_ROOT/'mask_hashes.json').is_file():
    assert json.loads((STUDY_ROOT/'mask_hashes.json').read_text())==mask_files
else:(STUDY_ROOT/'mask_hashes.json').write_text(json.dumps(mask_files,sort_keys=True))

b=hair_fast.blend;enc=b.blending_encoder;G=b.net.generator;pp=b.post_process;align=hair_fast.align
align.eval().requires_grad_(False)
BASE_SHA=sha(b.opts.blending_checkpoint)
assert BASE_SHA=='cece9608f2fe8269796a9903fb9550f2cb0e8022cedaefcbf9972305f206fed1'
assert sha(b.opts.pp_checkpoint)=='91ae95d3bf274a61d4d120daa955346033ce089c334221d5b444ff2ce92a6a84'
print('150 train + 9 validation | 4 x 30 epochs | รวม 18,000 updates | Alignment frozen')
display(pd.DataFrame(TRIALS))

EXPECTED_CACHE_HASHES_SHA='617de9ce1603ff990a40ba06a0ce923032c334b69c8bf7f4e8c4f383b658d16c'
SAVE_ALL_EPOCHS=False


In [ ]:
runtime_baseS={}
def cpu(x):
    if torch.is_tensor(x):
        return x.detach().cpu().clone()
    if isinstance(x, dict):
        return {k: cpu(v) for k, v in x.items()}
    if isinstance(x, list):
        return [cpu(v) for v in x]
    if isinstance(x, tuple):
        return tuple((cpu(v) for v in x))
    return copy.deepcopy(x)

def gpu(x):
    if torch.is_tensor(x):
        return x.to('cuda').clone()
    if isinstance(x, dict):
        return {k: gpu(v) for k, v in x.items()}
    if isinstance(x, list):return [gpu(v) for v in x]
    if isinstance(x, tuple):return tuple(gpu(v) for v in x)
    return x

def resize(x):
    return F.interpolate(x, size=(256, 256), mode='bilinear', align_corners=False)

def saveim(x, p):
    a = x.detach().float().cpu()[0].clamp(0, 1).permute(1, 2, 0).numpy()
    Image.fromarray((a * 255).round().astype('uint8')).save(p)

def lab(x):
    x = x.float().clamp(0, 1)
    lin = torch.where(x <= 0.04045, x / 12.92, ((x + 0.055) / 1.055).pow(2.4))
    M = x.new_tensor([[0.4124564, 0.3575761, 0.1804375], [0.2126729, 0.7151522, 0.072175], [0.0193339, 0.119192, 0.9503041]])
    xyz = torch.einsum('ij,bjhw->bihw', M, lin) / x.new_tensor([0.95047, 1.0, 1.08883])[None, :, None, None]
    f = torch.where(xyz > 0.008856, xyz.clamp_min(1e-08).pow(1 / 3), 7.787 * xyz + 16 / 116)
    return torch.stack([116 * f[:, 1] - 16, 500 * (f[:, 0] - f[:, 1]), 200 * (f[:, 1] - f[:, 2])], 1)

def render(s, d):
    S = torch.cat([d['S'][:, :6], s], 1)
    return G([S], input_is_latent=True, return_latents=False, start_layer=4, end_layer=8, layer_in=d['F'], randomize_noise=False)[0]

def finish(raw, d):
    sf, ff = pp(d['I1'], b.downsample_256(raw))
    return G([sf], input_is_latent=True, return_latents=False, start_layer=5, end_layer=8, layer_in=ff, randomize_noise=False)[0]

def detail(x):
    y = x.mean(1, keepdim=True)
    mu = F.avg_pool2d(y, 9, 1, 4)
    v = F.avg_pool2d((y - mu).square(), 9, 1, 4)
    return (y - mu) / (v + 0.001).sqrt()

def unit(raw):return resize(((raw+1)/2).clamp(0,1))
def read_mask(p):
    with Image.open(p) as im:
        if im.size!=(256,256):raise ValueError('Mask size mismatch')
        a=np.asarray(im.convert('L'))>127
    return torch.from_numpy(a.copy())[None,None]

def load_case(k):
    d=torch.load(SOURCE/'cache'/(k+'.pt'),map_location='cpu',weights_only=True)
    d['core']=d['core'].bool();d['refmask']=d['refmask'].bool();d['target']=d['target'].bool()
    d['eval_core']=d['core'].clone();d['eval_refmask']=d['refmask'].clone()
    if k in REVIEWED_KEYS:
        d['eval_core']=read_mask(MASKS/k/'color_core.png');d['eval_refmask']=read_mask(MASKS/k/'reference_core.png')
        assert torch.equal(d['target'],read_mask(MASKS/k/'target_unchanged.png'))
        assert not (d['eval_core']&~d['target']).any() and not (d['eval_core']&d['protected'].bool()).any()
    if k not in REVIEWED_KEYS:
        d['eval_core'] &= d['target'] & ~d['protected'].bool()
    for m in ['core','refmask','eval_core','eval_refmask']:
        assert d[m].shape==(1,1,256,256) and d[m].sum()>=100, (k,m)
    for im in ['face','ref','aligned']:
        assert d[im].shape==(1,3,256,256) and torch.isfinite(d[im]).all()
        assert d[im].min()>=-.001 and d[im].max()<=1.001
    if k in runtime_baseS:d['baseS']=runtime_baseS[k].clone()
    return d

def quantile_color(im,ref,mask,refmask):
    l=lab(im)[0,:,mask[0,0].bool()];r=lab(ref)[0,:,refmask[0,0].bool()]
    q=torch.linspace(.1,.9,9,device=im.device)
    diff=(torch.quantile(l,q,dim=1)-torch.quantile(r,q,dim=1)).abs()
    return diff[:,0].mean(),diff[:,1:].mean()

def masked_mae(a,b,m):return ((a-b).abs()*m.float()).sum()/(a.shape[1]*m.sum().clamp_min(1))

def shape_metrics(im,d):
    labels=segment(align.to_bisenet(im))
    pred=F.interpolate((labels==13).float().reshape(1,1,*labels.shape[-2:]),(256,256),mode='nearest')>.5
    target=d['target']
    return dict(target_mask_IoU=float((pred&target).sum()/(pred|target).sum().clamp_min(1)),
                target_missing_fraction=float((target&~pred).sum()/target.sum().clamp_min(1)))


# Frozen differentiable evaluators: reuse exact parser weights, load original ArcFace.
from models.CtrlHair.external_code.face_parsing.my_parsing_util import FaceParsing, FaceParsing_tensor
from losses.pp_losses import Backbone
from huggingface_hub import hf_hub_download
id_path=Path('pretrained_models/ArcFace/ir_se50.pth')
if not id_path.is_file():
    downloaded=hf_hub_download(repo_id='AIRI-Institute/HairFastGAN',filename=str(id_path))
    id_path.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(downloaded,id_path)
id_net=Backbone(input_size=112,num_layers=50,drop_ratio=.6,mode='ir_se')
id_net.load_state_dict(torch.load(id_path,map_location='cpu',weights_only=True),strict=True)
id_net=id_net.cuda().eval().requires_grad_(False)
FaceParsing_tensor.parsing_img()
parser_net=FaceParsing.bise_net
if parser_net is None:raise RuntimeError('Original BiSeNet is not initialized')
parser_net.eval().requires_grad_(False)
assert FaceParsing.label_list[17]=='hair' # raw parser label=17; remapped target label=13

def id_features(im):
    x=F.interpolate(im,size=(256,256),mode='bilinear',align_corners=False)*2-1
    x=F.adaptive_avg_pool2d(x[:,:,35:223,32:220],(112,112))
    feats=id_net(x)
    return F.normalize(feats,dim=1)

def constraint_terms(im,d,fx):
    # Do NOT call get_segmentation: its no_grad + argmax would sever training gradients.
    logits=checkpoint(lambda x:parser_net(align.to_bisenet(x))[0],im,use_reentrant=False) if torch.is_grad_enabled() else parser_net(align.to_bisenet(im))[0]
    other=torch.cat((logits[:,:17],logits[:,18:]),1)
    binary=logits[:,17:18]-torch.logsumexp(other,dim=1,keepdim=True)
    target=F.interpolate(d['target'].float(),size=binary.shape[-2:],mode='nearest')
    if target.sum()==0 or (1-target).sum()==0:raise RuntimeError('Degenerate shape target')
    prob=binary.sigmoid()
    bce=F.binary_cross_entropy_with_logits(binary,target,reduction='none')
    balanced=.5*((bce*target).sum()/target.sum()+(bce*(1-target)).sum()/(1-target).sum())
    dice=1-(2*(prob*target).sum()+1)/(prob.sum()+target.sum()+1)
    embedding=checkpoint(id_features,im,use_reentrant=False) if torch.is_grad_enabled() else id_features(im)
    identity=(1-(embedding*fx['face_embedding']).sum(1)).mean()
    return identity,balanced+dice,balanced,dice

def to_size(x,size):
    return F.interpolate(x,size=(size,size),mode='bilinear',align_corners=False)

def detail_terms(full,d,fx):
    values={}
    for size in [256,512]:
        x=to_size(full,size)
        teacher=fx['pre'] if size==256 else fx['pre_512']
        mask=F.interpolate(d['eval_core'].float(),size=(size,size),mode='nearest')
        inner=-F.max_pool2d(-mask,9,1,4)
        if inner.sum()<64:raise RuntimeError('Insufficient interior hair pixels at '+str(size))
        values[size]=masked_mae(detail(x),detail(teacher),inner)
    return values


from scipy.ndimage import binary_erosion,binary_dilation
from skimage.metrics import structural_similarity
_basic_shape_metrics=shape_metrics
def shape_metrics(im,d):
    result=_basic_shape_metrics(im,d)
    labels=segment(align.to_bisenet(im));p=F.interpolate((labels==13).float().reshape(1,1,*labels.shape[-2:]),(256,256),mode='nearest')[0,0].cpu().numpy()>.5
    t=d['target'][0,0].cpu().numpy()>.5
    result['target_Dice']=float(2*(p&t).sum()/max(p.sum()+t.sum(),1))
    a=p^binary_erosion(p);bnd=t^binary_erosion(t)
    prec=(a&binary_dilation(bnd,iterations=2)).sum()/max(a.sum(),1)
    rec=(bnd&binary_dilation(a,iterations=2)).sum()/max(bnd.sum(),1)
    result['boundary_F1_2px']=float(2*prec*rec/max(prec+rec,1e-12))
    return result

def extra_metrics(im,d):
    x=im[0].permute(1,2,0).cpu().numpy();y=d['face'][0].permute(1,2,0).cpu().numpy()
    mask=d['protected'][0,0].cpu().numpy()>.5
    _,ss=structural_similarity(x,y,data_range=1.,channel_axis=2,full=True)
    safe=binary_erosion(mask,iterations=3)
    if not safe.any():raise ValueError('No protected SSIM interior')
    mse=float(((x-y)**2)[mask].mean())
    return dict(protected_PSNR=float(-10*np.log10(max(mse,1e-12))),protected_SSIM=float(ss[safe].mean()),
        any_channel_255_fraction=float((im[0,:,d['eval_core'][0,0]].max(0).values>=1-0.5/255).float().mean()))


## ฟังก์ชันฝึกหนึ่ง trial
โหลด Blending ต้นฉบับ + PostProcess ต้นฉบับ ใหม่ทุกครั้ง ตรวจ fingerprint น้ำหนักเริ่มต้นและ hash cache. Validation ไม่มี backward. งบฝึกเท่ากันทุก trial.

In [ ]:
def run_trial(spec):
    global SOURCE
    BLEND_LR=spec['blend_lr'];PP_LR=spec['pp_lr']
    ROOT=STUDY_ROOT/('trial_'+spec['name']);ROOT.mkdir(exist_ok=True)
    RESUME_DIR=str(ROOT) if (ROOT/'last.pt').is_file() else ''
    
    # Train one joint model, no ablation arms and no inference-mode training wrapper.
    WEIGHTS=dict(final_color=1.,final_shape=.2,pre_color=.5,pre_shape=.1,
                 identity=.1,protected=5.,detail=.1,anchor=.01)
    # Fixed selection rule. Tolerances are operational safeguards, not proof of identity.
    GATES=dict(identity_drop=.02,protected_increase=.01,detail_relative_increase=.15)
    history=[];steps=[];errors=[];fixed={};status={}
    
    def fingerprint(module):
        h=hashlib.sha256()
        for name,t in module.state_dict().items():
            h.update(name.encode());h.update(t.detach().cpu().contiguous().numpy().tobytes())
        return h.hexdigest()
    
    def atomic_json(obj,p):
        temp=p.with_suffix(p.suffix+'.tmp')
        temp.write_text(json.dumps(obj,ensure_ascii=False,indent=2));os.replace(temp,p)
    
    def persist():
        for name,rows in [('metrics.csv',history),('training_steps.csv',steps)]:
            tmp=ROOT/(name+'.tmp');pd.DataFrame(rows).to_csv(tmp,index=False);os.replace(tmp,ROOT/name)
        atomic_json(status,ROOT/'manifest.json');atomic_json(errors,ROOT/'errors.json')
    
    def model_payload(epoch):
        return dict(format='hairfast_bpp_pretrained_start_trial_v1',epoch=epoch,
            blending_trainable_state_dict={n:p.detach().cpu().clone() for n,p in enc.named_parameters() if n.startswith('modulation_module_list.')},
            post_process_state_dict=cpu(pp.state_dict()),repo_commit=REPO_COMMIT,
            start_blending_sha256=START_SHA,mask_zip_sha256=MASK_ZIP_SHA,
            source_manifest_sha256=SOURCE_MANIFEST_SHA,config=config)
    
    def save_checkpoint(name,epoch,optimizer=None):
        payload=model_payload(epoch)
        if optimizer is not None:
            payload.update(optimizer=cpu(optimizer.state_dict()),history=history,steps=steps,status=copy.deepcopy(status))
        tmp=ROOT/(name+'.tmp');torch.save(payload,tmp);os.replace(tmp,ROOT/name)
        del payload
    
    def load_joint(ck):
        if ck.get('format')!='hairfast_bpp_pretrained_start_trial_v1' or ck['config']!=config:
            raise RuntimeError('Checkpoint/config ไม่ตรงกับรอบนี้')
        assert ck['repo_commit']==REPO_COMMIT and ck['mask_zip_sha256']==MASK_ZIP_SHA
        assert ck['source_manifest_sha256']==SOURCE_MANIFEST_SHA and ck['start_blending_sha256']==START_SHA
        state=enc.state_dict();sd=ck['blending_trainable_state_dict']
        assert set(sd)=={n for n,p in enc.named_parameters() if n.startswith('modulation_module_list.')}
        state.update(sd);enc.load_state_dict(state,strict=True)
        pp.load_state_dict(ck['post_process_state_dict'],strict=True)
    
    def joint_forward(d,grad=False):
        def blend_fn(a,c,x,y):return enc(a,c,x,y)
        args=[d['args'][i] for i in range(4)]
        s=checkpoint(blend_fn,*args,use_reentrant=False) if grad else blend_fn(*args)
        raw_pre=checkpoint(lambda a:render(a,d),s,use_reentrant=False) if grad else render(s,d)
        pp_in=b.downsample_256(raw_pre)
        sf,ff=checkpoint(lambda a,c:pp(a,c),d['I1'],pp_in,use_reentrant=False) if grad else pp(d['I1'],pp_in)
        def generate(a,c):return G([a],input_is_latent=True,return_latents=False,
            start_layer=5,end_layer=8,layer_in=c,randomize_noise=False)[0]
        raw=checkpoint(generate,sf,ff,use_reentrant=False) if grad else generate(sf,ff)
        return raw_pre,raw,s,sf,ff
    
    def objective(raw_pre,raw,s,sf,ff,d,fx):
        full=((raw+1)/2).clamp(0,1);im=to_size(full,256);pre=unit(raw_pre)
        L,ab=quantile_color(im,d['ref'],d['eval_core'],d['eval_refmask'])
        preL,preab=quantile_color(pre,d['ref'],d['eval_core'],d['eval_refmask'])
        identity,shape,_,_=constraint_terms(im,d,fx)
        _,preshape,_,_=constraint_terms(pre,d,fx)
        protected=masked_mae(im,d['face'],d['protected'])
        details=detail_terms(full,d,fx);texture=torch.stack(list(details.values())).mean()
        anchor=(s-fx['s']).square().mean()+(sf-fx['sf']).square().mean()+(ff-fx['ff']).square().mean()
        color=(L+ab)/100;precolor=(preL+preab)/100
        loss=(WEIGHTS['final_color']*color+WEIGHTS['final_shape']*shape+
              WEIGHTS['pre_color']*precolor+WEIGHTS['pre_shape']*preshape+
              WEIGHTS['identity']*identity+WEIGHTS['protected']*protected+
              WEIGHTS['detail']*texture+WEIGHTS['anchor']*anchor)
        return loss,dict(loss=loss,color_loss=color,shape_loss=shape,pre_color_loss=precolor,
            pre_shape_loss=preshape,identity_loss=identity,identity_cosine=1-identity,
            protected_RGB_MAE_vs_input=protected,detail_512_MAE_vs_pre=details[512],
            detail_MAE_vs_pre=details[256],detail_training_term=texture,latent_anchor=anchor,
            L_quantile=L,ab_quantile=ab,pre_L_quantile=preL,pre_ab_quantile=preab)
    
    def evaluate(epoch):
        pp.eval();enc.eval()
        # Validation only; training metrics are logged at each update.
        with torch.no_grad():
            for key in VAL_KEYS:
                d=gpu(load_case(key));fx=gpu(fixed[key])
                torch.cuda.synchronize();eval_tick=time.perf_counter()
                raw_pre,raw,s,sf,ff=joint_forward(d)
                torch.cuda.synchronize();forward_sec=time.perf_counter()-eval_tick
                _,vals=objective(raw_pre,raw,s,sf,ff,d,fx);im=unit(raw)
                vals.update(shape_metrics(im,d));vals.update(extra_metrics(im,d))
                vals.update({'pre_'+k:v for k,v in extra_metrics(unit(raw_pre),d).items()})
                vals['cached_forward_seconds']=forward_sec
                vals.update({'pre_'+k:v for k,v in shape_metrics(unit(raw_pre),d).items()})
                vals['near_white_fraction_L98']=(lab(im)[0,0,d['eval_core'][0,0]]>=98).float().mean()
                history.append(dict(epoch=epoch,case=key,split=bykey[key]['split'],**{k:float(v) for k,v in vals.items()}))
                folder=ROOT/'outputs'/f'epoch_{epoch:02d}'/key;folder.mkdir(parents=True,exist_ok=True)
                saveim((raw_pre+1)/2,folder/'pre.png');saveim((raw+1)/2,folder/'final.png')
                del d,fx,raw_pre,raw,s,sf,ff,vals,im
        frame=pd.DataFrame(history);val=frame[(frame.epoch==epoch)&(frame.split=='validation')].mean(numeric_only=True).to_dict()
        assert len(frame[(frame.epoch==epoch)&(frame.split=='validation')])==len(VAL_KEYS)
        persist()
        print('Validation epoch',epoch,': L',round(val['L_quantile'],3),'ab',round(val['ab_quantile'],3),
              'IoU',round(val['target_mask_IoU'],4),'ID',round(val['identity_cosine'],4),flush=True)
        return val
    
    def selection(val,baseline):
        color=val['L_quantile']+val['ab_quantile'];basecolor=baseline['L_quantile']+baseline['ab_quantile']
        shape=1-val['target_mask_IoU'];baseshape=1-baseline['target_mask_IoU']
        score=.5*(color/max(basecolor,1e-8)+shape/max(baseshape,1e-8))
        eligible=(color<=basecolor and shape<=baseshape and
            val['identity_cosine']>=baseline['identity_cosine']-GATES['identity_drop'] and
            val['protected_RGB_MAE_vs_input']<=baseline['protected_RGB_MAE_vs_input']+GATES['protected_increase'] and
            val['detail_512_MAE_vs_pre']<=baseline['detail_512_MAE_vs_pre']*(1+GATES['detail_relative_increase']))
        return score,bool(eligible)
    
    config=dict(trial=spec['name'],comparator_sha256=REFERENCE_SHA,init_blending_sha256=START_SHA,init_postprocess_sha256=PP_START_SHA,save_all_epochs=SAVE_ALL_EPOCHS,epochs=EPOCHS,blend_lr=BLEND_LR,pp_lr=PP_LR,seed=SEED,weights=WEIGHTS,gates=GATES,
                train_keys=TRAIN_KEYS,validation_keys=VAL_KEYS,batch_size=1,mask_policy='reviewed33_else_automatic_protected_excluded',reference_sha256=REFERENCE_SHA)
    assert EPOCHS==30 and set(TRAIN_KEYS).isdisjoint(VAL_KEYS)
    entry_enc=cpu(enc.state_dict());entry_pp=cpu(pp.state_dict())
    states=[(m,[(sub,sub.training) for sub in m.modules()],[(p,p.requires_grad) for p in m.parameters()]) for m in [enc,G,pp]]
    opt=None
    try:
        for m,_,_ in states:m.eval();m.requires_grad_(False)
        # Reset to official pretrained Blending + official pretrained PP for EACH trial, without any user fine-tuned starting weights.
        base_sd=torch.load(b.opts.blending_checkpoint,map_location='cpu',weights_only=True)['model_state_dict']
        missing,unexpected=enc.load_state_dict(base_sd,strict=False)
        if unexpected or any(not k.startswith('clip_model.') for k in missing):raise RuntimeError('Blending keys mismatch')
        expected={n for n,p in enc.named_parameters() if n.startswith('modulation_module_list.')}
        assert set(REFERENCE['blending_trainable_state_dict'])==expected # comparator schema only
        pp.load_state_dict(torch.load(b.opts.pp_checkpoint,map_location='cpu',weights_only=True)['model_state_dict'],strict=True)
        del base_sd
        initial_fingerprints={'blending':fingerprint(enc),'post_process':fingerprint(pp)}
        init_path=STUDY_ROOT/'initial_fingerprints.json'
        if init_path.is_file():assert json.loads(init_path.read_text())==initial_fingerprints,'Trial starting weights differ'
        else:atomic_json(initial_fingerprints,init_path)
        source_hashes={k:sha(SOURCE/'cache'/(k+'.pt')) for k in KEYS}
        assert hashlib.sha256(json.dumps(source_hashes,sort_keys=True).encode()).hexdigest()==EXPECTED_CACHE_HASHES_SHA,'Full150 cache bytes changed'
    
        # Copy verified cache locally.
        ORIGINAL_SOURCE=SOURCE
        LOCAL_SOURCE=Path('/content')/('hfg_joint_cache_'+ROOT.name);(LOCAL_SOURCE/'cache').mkdir(parents=True,exist_ok=True)
        for k in KEYS:
            dst=LOCAL_SOURCE/'cache'/(k+'.pt')
            if not dst.is_file() or sha(dst)!=source_hashes[k]:shutil.copy2(SOURCE/'cache'/(k+'.pt'),dst)
        SOURCE=LOCAL_SOURCE
        frozen_before={n:fingerprint(m) for n,m in [('alignment',align),('generator',G),('clip',enc.clip_model),('arcface',id_net),('parser',parser_net)]}
        random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
        status=dict(status='preparing',trial=spec['name'],comparator_checkpoint=str(REFERENCE_PT),starting_blending_checkpoint=str(START),starting_postprocess_checkpoint=str(b.opts.pp_checkpoint),comparator_sha256=REFERENCE_SHA,init_blending_sha256=START_SHA,init_postprocess_sha256=PP_START_SHA,training=True,weight_updates=0,config=config,source=str(ORIGINAL_SOURCE),
            source_cache_hashes=source_hashes,repo_commit=REPO_COMMIT,
            trainable=['blending.modulation_module_list','all post_process parameters'],
            frozen=['generator','clip','parser','arcface','inversion','alignment'],
            shape_target='cached automatic target mask; assistant thumbnail screening only, not pixel GT',
            mask_policy=config['mask_policy'],reference_sha256=REFERENCE_SHA,audit_zip_sha256=AUDIT_ZIP_SHA,
            selection='validation mean color and shape normalized to epoch0; face and detail gates',
            cache_policy='shape rotation OFF; color rotation as cached; mix 0.95; no re-alignment',
            best_epoch=0,best_score=1.,selection_history=[])
        # Baseline pseudo-detail targets are fixed once, never rebuilt from resumed weights.
        with torch.no_grad():
            for key in KEYS:
                folder=ROOT/'inputs'/key;folder.mkdir(parents=True,exist_ok=True);fp=folder/'fixed_inputs.pt'
                if RESUME_DIR:
                    if not fp.is_file():raise FileNotFoundError('Baseline cache missing '+str(fp))
                    fixed[key]=torch.load(fp,map_location='cpu',weights_only=True)
                    continue
                d=gpu(load_case(key));raw_pre,raw,s,sf,ff=joint_forward(d)
                fixed[key]=cpu(dict(pre=unit(raw_pre),pre_512=to_size(((raw_pre+1)/2).clamp(0,1),512),
                    s=s,sf=sf,ff=ff,face_embedding=id_features(d['face'])))
                tmp=fp.with_suffix('.tmp');torch.save(fixed[key],tmp);os.replace(tmp,fp)
                for name in ['face','ref','aligned']:saveim(d[name],folder/(name+'.png'))
                saveim((raw_pre+1)/2,folder/'pre.png');saveim((raw+1)/2,folder/'old_final.png')
                for name in ['eval_core','eval_refmask','target','protected']:
                    Image.fromarray((d[name][0,0].cpu().numpy()*255).astype('uint8')).save(folder/(name+'.png'))
                del d,raw_pre,raw,s,sf,ff
                print('เตรียมฐาน',key,flush=True)
        for n,p in enc.named_parameters():p.requires_grad_(n.startswith('modulation_module_list.'))
        pp.requires_grad_(True)
        bp=[p for p in enc.parameters() if p.requires_grad];pparams=list(pp.parameters());params=bp+pparams
        opt=torch.optim.Adam([{'params':bp,'lr':BLEND_LR},{'params':pparams,'lr':PP_LR}])
        if RESUME_DIR:
            ck=torch.load(ROOT/'last.pt',map_location='cpu',weights_only=False) # only our own trusted checkpoint
            load_joint(ck)
            if ck['status']['source_cache_hashes']!=source_hashes:raise RuntimeError('Source cache changed')
            opt.load_state_dict(ck['optimizer']);history=ck['history'];steps=ck['steps'];status=ck['status']
            start_epoch=ck['epoch']+1;del ck
            print('รันต่อจาก epoch',start_epoch,'ใช้ log/checkpoint เดิม',flush=True)
        else:
            # Evaluate old epoch27 under the exact same masks and fixed loss targets.
            # Preserve original initialization; comparator weights NEVER enter training.
            init_enc=cpu(enc.state_dict());init_pp=cpu(pp.state_dict())
            try:
                state=enc.state_dict();state.update(REFERENCE['blending_trainable_state_dict'])
                enc.load_state_dict(state,strict=True);pp.load_state_dict(REFERENCE['post_process_state_dict'],strict=True)
                refval=evaluate(-27)
                status['epoch27_validation']=refval
            finally:
                enc.load_state_dict(init_enc,strict=True);pp.load_state_dict(init_pp,strict=True)
            del init_enc,init_pp,state
            assert fingerprint(enc)==initial_fingerprints['blending']
            assert fingerprint(pp)==initial_fingerprints['post_process']
            val=evaluate(0);status['baseline_validation']=val
            status['epoch27_score_vs_start'],status['epoch27_eligible_vs_start']=selection(refval,val)
            save_checkpoint('best.pt',0);save_checkpoint('last.pt',0,opt);start_epoch=1
        if start_epoch>EPOCHS:print('รอบนี้ครบ 30 epochs แล้ว จะส่งออกผลเดิม')
        # One differentiability check on a TRAIN sample; not another experiment.
        if start_epoch<=EPOCHS:
            key=TRAIN_KEYS[0];d=gpu(load_case(key));fx=gpu(fixed[key])
            raw_pre,raw,s,sf,ff=joint_forward(d,True);loss,vals=objective(raw_pre,raw,s,sf,ff,d,fx)
            probe={}
            for term in ['color_loss','shape_loss','pre_color_loss','pre_shape_loss']:
                out=raw_pre if term.startswith('pre_') else raw
                grad=torch.autograd.grad(vals[term],out,retain_graph=True)[0]
                if not torch.isfinite(grad).all() or grad.norm()<=0:raise RuntimeError('Invalid gradient '+term)
                probe[term]=float(grad.norm());del grad
            loss.backward()
            for label,group in [('blending',bp),('post_process',pparams)]:
                grads=[p.grad for p in group if p.grad is not None]
                if not grads or not all(torch.isfinite(g).all() for g in grads):raise RuntimeError('Invalid gradients '+label)
                probe[label+'_grad_norm']=sum(float(g.float().square().sum()) for g in grads)**.5
                if probe[label+'_grad_norm']<=0:raise RuntimeError('Zero gradients '+label)
            del grads
            status['gradient_probe']=probe;opt.zero_grad(set_to_none=True)
            del d,fx,raw_pre,raw,s,sf,ff,loss,vals;gc.collect();torch.cuda.empty_cache();persist()
        status['status']='running'
        for epoch in range(start_epoch,EPOCHS+1):
            pp.eval();enc.eval() # update weights, retain pretrained BatchNorm statistics
            # Deterministic per-epoch shuffle makes epoch-boundary resume reproducible.
            random.seed(SEED+epoch);np.random.seed(SEED+epoch);torch.manual_seed(SEED+epoch);torch.cuda.manual_seed_all(SEED+epoch)
            order=TRAIN_KEYS.copy();random.Random(SEED+epoch).shuffle(order)
            epoch_tick=time.perf_counter()
            for key in order:
                assert bykey[key]['split']=='train'
                d=gpu(load_case(key));fx=gpu(fixed[key]);opt.zero_grad(set_to_none=True)
                torch.cuda.reset_peak_memory_stats();torch.cuda.synchronize();tick=time.perf_counter()
                raw_pre,raw,s,sf,ff=joint_forward(d,True);loss,vals=objective(raw_pre,raw,s,sf,ff,d,fx)
                if not loss.requires_grad or not torch.isfinite(loss):raise RuntimeError('Invalid loss')
                loss.backward();grads=[p.grad for p in params if p.grad is not None]
                if not grads or not all(torch.isfinite(g).all() for g in grads):raise RuntimeError('Nonfinite gradient')
                norm=torch.nn.utils.clip_grad_norm_(params,1.)
                if not torch.isfinite(norm) or norm<=0:raise RuntimeError('Invalid gradient norm')
                if any(p.grad is not None for m in [G,enc.clip_model,id_net,parser_net] for p in m.parameters()):raise RuntimeError('Frozen weights have gradients')
                opt.step();torch.cuda.synchronize()
                steps.append(dict(epoch=epoch,case=key,update=len(steps)+1,seconds=time.perf_counter()-tick,
                    gradient_norm=float(norm),peak_vram_MB=torch.cuda.max_memory_allocated()/1024**2,
                    **{k:float(v.detach()) for k,v in vals.items()}))
                status['weight_updates']=len(steps);persist()
                print(f"epoch {epoch}/{EPOCHS} | {key} | update {len(steps)}/{len(TRAIN_KEYS)*EPOCHS} | loss {float(loss.detach()):.5f}",flush=True)
                del d,fx,raw_pre,raw,s,sf,ff,loss,vals,grads
            opt.zero_grad(set_to_none=True)
            val=evaluate(epoch);score,eligible=selection(val,status['baseline_validation'])
            status['selection_history'].append(dict(epoch=epoch,score=score,eligible=eligible))
            if eligible and score<status['best_score']:
                status['best_score']=score;status['best_epoch']=epoch;save_checkpoint('best.pt',epoch)
            status['completed_epoch']=epoch
            # Retain best/last; optionally all epoch weights. Optimizer is retained in last.pt.
            if SAVE_ALL_EPOCHS:save_checkpoint(f'epoch_{epoch:02d}.pt',epoch)
            save_checkpoint('last.pt',epoch,opt);persist()
            elapsed=time.perf_counter()-epoch_tick
            print(f"จบ epoch {epoch} | {elapsed/60:.1f} นาที | ประมาณเวลาที่เหลือ {(EPOCHS-epoch)*elapsed/3600:.2f} ชม. | best epoch {status['best_epoch']}",flush=True)
        status['frozen_unchanged']={n:fingerprint(m)==frozen_before[n] for n,m in [('alignment',align),('generator',G),('clip',enc.clip_model),('arcface',id_net),('parser',parser_net)]}
        assert all(status['frozen_unchanged'].values())
        assert len(steps)==EPOCHS*len(TRAIN_KEYS)
        save_checkpoint('final_epoch30.pt',EPOCHS)
        status['status']='complete';persist()
    except BaseException as e:
        status['status']='interrupted' if isinstance(e,KeyboardInterrupt) else 'error'
        errors.append(dict(message=str(e),traceback=traceback.format_exc()));print(traceback.format_exc())
        print('last.pt เก็บ epoch ที่สำเร็จล่าสุดไว้แล้ว ใส่ STUDY_ROOT ใน RESUME_STUDY_DIR แล้วเปิด runtime ใหม่เพื่อรันต่อ')
    finally:
        opt=None;gc.collect()
        enc.load_state_dict(entry_enc,strict=True);pp.load_state_dict(entry_pp,strict=True)
        for m,modes,flags in states:
            for p,flag in flags:p.requires_grad_(flag)
            for sub,mode in modes:sub.training=mode
        if 'ORIGINAL_SOURCE' in locals():SOURCE=ORIGINAL_SOURCE
        status['entry_weights_restored']=True;persist()
    print('สถานะ:',status.get('status'),'| updates:',len(steps),'| ผล:',ROOT)
    
    if status.get('status')!='complete':
        raise RuntimeError('Trial '+spec['name']+' stopped: '+(errors[-1]['message'] if errors else str(status)))
    return dict(trial=spec['name'],root=str(ROOT),best_epoch=status['best_epoch'],best_score=status['best_score'])


## รัน A → B → C → D และส่งออกรายงาน
ภาพเปรียบเทียบใช้ input / shape-aligned / color reference ของ cache เดิม; shape-aligned ไม่ใช่ภาพเรฟทรงดิบ. ตรวจภาพกับเรฟทรงดิบเพิ่มเติมเมื่อตัดสินรายละเอียดลอน.

In [ ]:
def compare_to_epoch27(val,ref):
    color=val['L_quantile']+val['ab_quantile'];rc=ref['L_quantile']+ref['ab_quantile']
    shape=1-val['target_mask_IoU'];rs=1-ref['target_mask_IoU']
    score=.5*(color/max(rc,1e-8)+shape/max(rs,1e-8))
    eligible=(color<=rc and shape<=rs and val['identity_cosine']>=ref['identity_cosine']-.02
        and val['protected_RGB_MAE_vs_input']<=ref['protected_RGB_MAE_vs_input']+.01
        and val['detail_512_MAE_vs_pre']<=ref['detail_512_MAE_vs_pre']*1.15)
    return dict(score_vs_epoch27=float(score),improved_over_epoch27=bool(eligible and score<1))

def export_study_report():
    review=STUDY_ROOT/'review';review.mkdir(exist_ok=True)
    rows=[]
    for spec in TRIALS:
        root=STUDY_ROOT/('trial_'+spec['name'])
        if not (root/'manifest.json').is_file() or not (root/'metrics.csv').is_file():continue
        m=json.loads((root/'manifest.json').read_text())
        frame=pd.read_csv(root/'metrics.csv')
        if frame.empty:continue
        summary=frame[frame.split=='validation'].groupby('epoch').mean(numeric_only=True)
        summary.to_csv(root/'summary.csv')
        best=int(m.get('best_epoch',0))
        if best not in summary.index:continue
        values=summary.loc[best].to_dict()
        steps_frame=pd.read_csv(root/'training_steps.csv') if (root/'training_steps.csv').is_file() and (root/'training_steps.csv').stat().st_size>2 else pd.DataFrame()
        rows.append(dict(trial=spec['name'],status=m['status'],blend_lr=spec['blend_lr'],pp_lr=spec['pp_lr'],
            best_epoch=best,score=m['best_score'],improved_over_start=(best>0 and m['best_score']<1),
            updates=m.get('weight_updates',0),training_step_hours=float(steps_frame['seconds'].sum()/3600) if len(steps_frame) else 0,
            checkpoint=str(root/'best.pt'),**compare_to_epoch27(values,m['epoch27_validation']),**values))
        dest=review/('trial_'+spec['name']);dest.mkdir(exist_ok=True)
        for name in ['manifest.json','errors.json','metrics.csv','training_steps.csv','summary.csv']:
            if (root/name).is_file():shutil.copy2(root/name,dest/name)
        for key in VAL_KEYS:
            inp=root/'inputs'/key
            panels=[(label,inp/(name+'.png')) for label,name in [('Input','face'),('Cached aligned','aligned'),('Color reference','ref')]]
            for ep in sorted({-27,0,best,int(summary.index.max())}):
                panels.extend([(('old ep27 pre' if ep==-27 else f'ep{ep} pre'),root/'outputs'/f'epoch_{ep:02d}'/key/'pre.png'),
                               (('old ep27 final' if ep==-27 else f'ep{ep} final'),root/'outputs'/f'epoch_{ep:02d}'/key/'final.png')])
            panels=[(label,p) for label,p in panels if p.is_file()]
            canvas=Image.new('RGB',(768,((len(panels)+2)//3)*280),'white');draw=ImageDraw.Draw(canvas)
            for j,(label,p) in enumerate(panels):
                x=j%3*256;y=j//3*280;draw.text((x+4,y+3),label,fill='black')
                with Image.open(p) as im:canvas.paste(im.convert('RGB').resize((256,256)),(x,y+24))
            canvas.save(dest/(key+'.jpg'),quality=92)
    if not rows:return None
    board=pd.DataFrame(rows).sort_values(['score','trial'])
    board.to_csv(STUDY_ROOT/'leaderboard.csv',index=False);board.to_csv(review/'leaderboard.csv',index=False)
    comparator_rows=[]
    for spec in TRIALS:
        mp=STUDY_ROOT/('trial_'+spec['name'])/'manifest.json'
        if mp.is_file():
            mm=json.loads(mp.read_text())
            if 'epoch27_validation' in mm:
                comparator_rows.append(dict(evaluated_in_trial=spec['name'],model='old_Full150_epoch27',**mm['epoch27_validation']))
    pd.DataFrame(comparator_rows).to_csv(review/'epoch27_comparator.csv',index=False)
    display(board[['trial','status','best_epoch','score','improved_over_epoch27','L_quantile','ab_quantile',
                   'target_mask_IoU','identity_cosine','protected_RGB_MAE_vs_input','training_step_hours']])
    complete=board[(board.status=='complete')&(board.updates==EPOCHS*len(TRAIN_KEYS))]
    if len(complete)==len(TRIALS):
        eligible=complete[complete.improved_over_epoch27].sort_values(['score_vs_epoch27','trial'])
        if len(eligible):
            r=eligible.iloc[0];winner=dict(trial=str(r.trial),epoch=int(r.best_epoch),score_vs_epoch27=float(r.score_vs_epoch27),checkpoint=str(r.checkpoint),
                basis='validation metrics; inspect images before deployment; ties need seed repeats')
        else:winner=dict(trial='old_Full150_epoch27',epoch=27,score_vs_epoch27=1.,checkpoint=str(REFERENCE_PT),basis='no new trial passed selection')
        (STUDY_ROOT/'winner.json').write_text(json.dumps(winner,indent=2));shutil.copy2(STUDY_ROOT/'winner.json',review/'winner.json')
        print('ผู้ชนะตาม validation:',winner)
    else:print('ยังไม่ครบ 4 trials ตารางนี้เป็นผลชั่วคราว ยังไม่เลือกผู้ชนะทั้งการทดลอง')
    shutil.copy2(STUDY_ROOT/'study.json',review/'study.json')
    archive=STUDY_ROOT/'bpp_4trials_pretrained_start_review.zip'
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in review.rglob('*'):
            if p.is_file():z.write(p,p.relative_to(review))
    return archive

try:
    for spec in TRIALS:
        root=STUDY_ROOT/('trial_'+spec['name']);mp=root/'manifest.json'
        if mp.is_file():
            previous=json.loads(mp.read_text());cfg=previous['config']
            assert cfg['trial']==spec['name'] and cfg['blend_lr']==spec['blend_lr'] and cfg['pp_lr']==spec['pp_lr']
            assert cfg['init_blending_sha256']==START_SHA
            assert cfg['comparator_sha256']==REFERENCE_SHA and cfg['epochs']==EPOCHS and cfg['seed']==SEED
            if previous['status']=='complete':
                assert previous['weight_updates']==EPOCHS*len(TRAIN_KEYS) and (root/'best.pt').is_file()
                print('ข้าม trial ที่เสร็จแล้ว:',spec['name']);continue
        print('เริ่ม trial',spec['name'],'จาก official pretrained Blending + official pretrained PP ด้วย LR',spec['blend_lr'],spec['pp_lr'],flush=True)
        run_trial(spec)
        gc.collect();torch.cuda.empty_cache()
        export_study_report()
finally:
    archive=export_study_report()
    if archive:
        print('รายงาน:',archive);files.download(str(archive))
print('เสร็จครบ 4 trials:',STUDY_ROOT)


## โหลดผู้ชนะ (ทางเลือก)
checkpoint มี Blending modulation และ PostProcess; ใช้ Alignment เดิม. ผลเว็บที่จัดแนวต่างจาก cache ยังต้องทดสอบแยก.

In [ ]:
USE_WINNER=False
if USE_WINNER:
    if not (STUDY_ROOT/'winner.json').is_file():raise FileNotFoundError('ยังไม่มีรายงานผู้ชนะ')
    selected_info=json.loads((STUDY_ROOT/'winner.json').read_text())
    ck=torch.load(selected_info['checkpoint'],map_location='cpu',weights_only=True)
    state=enc.state_dict();state.update(ck['blending_trainable_state_dict']);enc.load_state_dict(state,strict=True)
    pp.load_state_dict(ck['post_process_state_dict'],strict=True)
    enc.eval();pp.eval()
    print('Loaded:',selected_info)


## ข้อจำกัด
นี่เป็นการค้นหาบน validation 9 คู่เท่านั้น ยังไม่ใช้ test_candidate. ควรตรวจภาพผู้ชนะ และถ้าคะแนนใกล้กันค่อยทดลอง seed อื่นด้วยงบเท่ากัน.
คะแนนและผลของ live Alignment ในสองรอบล่าสุดใช้ preprocessing/mask ต่างกัน จึงไม่นำมาเข้าตารางจัดอันดับนี้.
Source: Notebook Full150 เดิมของผู้ใช้, HairFastGAN commit 49e98019c9243d1991c19d45764fc3340a498de2.
